# Prediksi Dataset Daging Ayam Ras

## Import Library

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold
import random
import tensorflow as tf

## Data Cleaning

In [2]:
# Set seed 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
df = pd.read_csv("train/Daging Ayam Ras.csv")

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

In [ ]:
numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())

## Data Pre-Processing

In [ ]:
def df_to_X_y(df, window_size=5):
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size])
        y.append(df[i + window_size])
    return np.array(X), np.array(y)

In [ ]:
df = df.select_dtypes(include=[np.number])
df = df.apply(pd.to_numeric, errors='coerce').dropna()

In [ ]:
scaler = StandardScaler()
df = scaler.fit_transform(df.values.reshape(-1, 1))

In [ ]:
window_size = 5
X, y = df_to_X_y(df, window_size)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False, random_state=SEED)

## Modelling & Evaluation

In [ ]:
checkpoint_path = "model_checkpoint.keras"
cp4 = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.8, 
    patience=5, 
    min_lr=1e-6, 
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,  
    restore_best_weights=True,
    verbose=1
)

In [ ]:
def create_lstm_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=input_shape),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(32, return_sequences=False),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='linear')
    ])
    model.compile(loss='mape', optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), metrics=['mape'])
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
model = create_lstm_model(input_shape)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), 
          epochs=100, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Final epoch:", len(history.history['loss']))

model.save("lstm_model.keras")

df_submission = pd.read_csv("sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)
print("required predictions:", total_required_predictions)
print("countries:", len(unique_countries))

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, window_size, 1))[0, 0]
    np.random.seed(SEED)  
    pred += np.random.normal(0, 0.01)  
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

y_pred = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

data = []
for i in range(min(len(df_submission), len(y_pred))):
    data.append({'id': df_submission.iloc[i]['id'], 'price': y_pred[i][0]})

submission_df = pd.DataFrame(data)
submission_df.to_csv("daging_ayam_ras_submission.csv", index=False)

In [ ]:
input_file = "daging_ayam_ras_submission.csv"  
output_file = "daging_ayam_ras_submission.csv"  

with open(input_file, "r", encoding="utf-8") as file:
    content = file.read()

updated_content = content.replace("Bawang Merah", "Daging Ayam Ras")

with open(output_file, "w", encoding="utf-8") as file:
    file.write(updated_content)

print("File updated")